# Datathon 7MLET — EDA e replay (y real)

Caso de uso: escolher **canal de contato** (`cellular` vs `telephone`) para campanha de depósito a prazo.

Fonte: [Kaggle henriqueyamahata/bank-marketing](https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing) = UCI Bank Marketing (`bank-additional-full.csv`). Este notebook usa o recorte real `tests/fixtures/bank_sample.csv` (linhas 11600:12800 do full). **Drop `duration`.** Sem cliente sintético.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import CONTACT_COL, FIXTURE_CSV, TARGET_COL, arm_rates, golden_rows, load_raw, prepare
from src.replay import run_comparison
from src.state import save_state
from src.tracking import log_replay

raw = load_raw(FIXTURE_CSV)
print("linhas", len(raw), "colunas", list(raw.columns))
print("duration presente no bruto?", "duration" in raw.columns)
print(raw["contact"].value_counts())
print(raw["y"].value_counts())

In [ ]:
df = prepare(raw)
assert "duration" not in df.columns
rates = arm_rates(df)
rates

## Replay: baseline sempre telephone vs Epsilon-Greedy

Recompensa = `y` da linha, **somente** se o braço escolhido for o `contact` logado. Nada de Bernoulli.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

base, eg_result, eg = run_comparison(df, epsilon=0.1, seed=42)
save_state(eg, {"baseline": base.as_metrics(), "epsilon_greedy": eg_result.as_metrics()})
run_id = log_replay(base, eg_result, epsilon=0.1, seed=42)

summary = pd.DataFrame([base.as_metrics(), eg_result.as_metrics()])
print("mlflow run_id", run_id)
print("médias EG", eg.means())
print("taxa telephone na tabela", float(df.loc[df[CONTACT_COL]=="telephone", TARGET_COL].mean()))
print("conversão baseline (deve coincidir)", base.conversion)
summary

In [ ]:
def cumulative(ys):
    ys = np.asarray(ys, dtype=float)
    return np.cumsum(ys) / np.arange(1, len(ys) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cumulative(base.rewards), label=f"baseline telephone ({base.conversion:.3f})")
ax.plot(cumulative(eg_result.rewards), label=f"epsilon-greedy ({eg_result.conversion:.3f})")
ax.set_xlabel("Rodadas aceitas no replay")
ax.set_ylabel("Conversão acumulada")
ax.set_title("Replay com y real — fixture UCI")
ax.legend()
ax.grid(True, alpha=0.3)
fig

In [ ]:
golden_rows(df)[["golden_index", "age", "job", "contact", "month", "campaign", "y"]]